In [1]:
# --- Core Python ---
import os

# --- Numerical & Data Analysis ---
import numpy as np
import pandas as pd

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns
import re

# --- Signal Processing ---
from scipy.signal import butter, filtfilt, resample, resample_poly, find_peaks

# --- Statistics ---
from scipy.stats import wilcoxon

# --- Specialized Neuro Tools ---
import neurokit2 as nk

print("✅ Imports loaded")

# --- Pandas display settings (optional) ---
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)

✅ Imports loaded


In [ ]:
# --- Respiration file paths (.h5) --- # Baseline recordings
resp_paths_bl = {
    "BL_1_1_d1_2": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_1_d1_2_20250623_103713_merged.h5",
    "BL_1_2_sub1_1": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_2_sub_1_1_20250623_120135_merged.h5",
    "BL_2_3_d2_4": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_3_d2_4_20250623_145448_merged.h5",
    "BL_2_4_sub2_3": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_4_sub2_3_20250623_141419_merged.h5",
    "BL_3_5_d3_6": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_5_d3_6_20250623_154154_merged.h5",
    "BL_3_6_sub3_5": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_6_sub3_5_20250623_172635_merged.h5",
    "BL_4_7_d4_8": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_7_d4_8_20250623_185042_merged.h5",
    "BL_4_8_sub4_7": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_8_sub4_7_20250623_180810_merged.h5"
}
resp_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_1_d1_2_20250623_111352_merged.h5",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_2_sub1_1_20250623_133932_merged.h5",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_3_d2_4_20250623_151153_merged.h5",
    "CM_2_4_sub2_2": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_4_sub2_3_20250623_143348_merged.h5",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_5_d3_6_20250623_170708_merged.h5",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_6_sub3_5_20250623_174348_merged.h5",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_7_d4_8_20250623_193718_merged.h5",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_8_sub4_7_20250623_182649_merged.h5",
}
# --- BORIS annotation file paths (.csv) --- # Baseline recordings
boris_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_1_d1_2_20250623_111352.1.csv",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_2_sub1_1_20250623_133932.1_VT.csv",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_3_d2_4_20250623_151153.1.csv",
    "CM_2_4_sub2_3": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_4_sub2_3_20250623_143348.1_VT.csv",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_5_d3_6_20250623_160001.csv",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_6_sub3_5_20250623_174348.csv",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_7_d4_8_20250623_193718.csv",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_8_sub4_7_20250623_182649.1_VT.csv",
}
print(f"Resp files: {len(resp_paths_cm)} trials")
print(f"Resp files: {len(resp_paths_bl)} trials")
print(f"BORIS files: {len(boris_paths_cm)} trials")

Resp files: 8 trials
Resp files: 8 trials
BORIS files: 8 trials


In [4]:
# --- Respiration file paths (.h5) ---
resp_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_7_p5_2_nRB3_20250621_150707_merged.h5",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_3_p5_3_nRB3_20250622_104059_merged (1).h5",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_3_p5_3_nRB3_20250622_110216_merged.h5",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_8_p5_1_nRB3_20250621_165214_merged.h5",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_8_p5_1_nRB3_20250621_171318_merged.h5",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_1_p5_2_nRB6_20250622_143958_merged.h5",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_1_p5_2_nRB6_20250622_150457_merged.h5",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_2_p5_1_nRB6_20250622_170742_merged.h5",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_2_p5_1_nRB6_20250622_173049_merged.h5",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_4_p5_4_nRB3_20250622_123424_merged.h5",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_4_p5_4_nRB3_20250622_125648_merged.h5",
    "RI1_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_5_p5_4_nRB3_20250621_105014_merged.h5",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_5_p5_4_nRB3_20250621_112618_merged.h5",
    # Baseline recordings
    "BLRI_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_1_p5_2_nRB6_20250622_141846_merged.h5",
    "BLRI_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_2_p5_1_nRB6_20250622_164833_merged.h5",
    "BLRI_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_3_p5_3_nRB3_20250622_101813_merged.h5",
    "BLRI_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_4_p5_4_nRB3_20250622_121708_merged.h5",
    "BLRI_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s3_6_p5_3_nRB3_20250621_123634_merged.h5",
    "BLRI_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_7_p5_2_nRB3_20250621_144806_merged.h5",
    "BLRI_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_8_p5_1_nRB3_20250621_163506_merged.h5",
}

# --- BORIS annotation file paths (.csv) ---
boris_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s3_6_p5_3_nRB3_HEEPS.csv",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_6_p_5_3_nRB3_2025062.csv",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_3_p5_3_nRB3_20250622_104059.1.csv",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_3_p5_3_nRB3_20250622_1102116.1.csv",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_8_p5_1_nRB3_HEEPS.csv",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_8_p5_1_nRB3_20250621_HEEPS.csv",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_1_p5_2_nRB6_20250622_143958.1.csv",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_1_p5_2_nRB6_20250622_150457.1.csv",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_2_p5_1_nRB6_20250622_170742.1.csv",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_2_p5_1_nRB6_20250622_173049.1.csv",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_4_p5_4_nRB3_20250622_123424.1.csv",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_4_p5_4_nRB3_20250622_125648.1.csv",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_5_p5_4_nRB3_20250621.csv",
}

print(f"Resp files: {len(resp_paths)} trials")
print(f"BORIS files: {len(boris_paths)} trials")

Resp files: 23 trials
BORIS files: 15 trials


In [12]:
import pandas as pd

def load_clean_boris(csv_path):
    """
    Load a BORIS CSV file and standardize columns for behavior alignment.

    Returns
    -------
    df : DataFrame
        Columns: ['Behavior', 'Subject', 'Start', 'Stop', 'Duration']
        Keeps only 'subject' rows and social behaviors.
    """
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"⚠️ Could not load {csv_path}: {e}")
        return pd.DataFrame()

    # --- Detect and rename possible column variants ---
    rename_map = {
        "Behavior": "Behavior",
        "Subject": "Subject",
        "Start (s)": "Start",
        "Stop (s)": "Stop",
        "Duration (s)": "Duration",
        "Start": "Start",
        "Stop": "Stop",
        "Duration": "Duration"
    }
    df = df.rename(columns=rename_map)

    # --- Keep essential columns only ---
    keep_cols = [c for c in ["Behavior", "Subject", "Start", "Stop", "Duration"] if c in df.columns]
    df = df[keep_cols].copy()

    # --- Clean ---
    df = df.dropna(subset=["Behavior", "Subject", "Start", "Stop"])
    df["Behavior"] = df["Behavior"].str.lower().str.strip()
    df["Subject"] = df["Subject"].str.lower().str.strip()

    # --- Keep only subject-initiated behaviors ---
    df = df[df["Subject"] == "subject"]

    # --- Focus on relevant social behaviors ---
    behaviors_keep = ["facial sniffing", "body sniffing", "anogenital sniffing"]
    df = df[df["Behavior"].isin(behaviors_keep)]

    # --- Sort chronologically ---
    df = df.sort_values("Start").reset_index(drop=True)

    return df


In [13]:
boris_path = r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_2_p5_1_nRB6_20250622_170742.1.csv"
boris_df = load_clean_boris(boris_path)

print(boris_df.head())


              Behavior  Subject   Start    Stop  Duration
0        body sniffing  subject  16.155  18.200     2.045
1        body sniffing  subject  19.300  21.767     2.467
2        body sniffing  subject  23.833  25.100     1.267
3  anogenital sniffing  subject  25.567  26.867     1.300
4        body sniffing  subject  27.233  27.833     0.600


In [6]:
def load_clean_resp_signal(h5_file, target_rate=100):
    """
    Loads, filters, downsamples respiration from .h5, returns cleaned signal, time vector, and metadata.
    """
    try:
        with h5py.File(h5_file, 'r') as f:
            resp = f['resp'][:].flatten()

            # Load metadata if available
            metadata = {}
            if 'resp_metadata' in f:
                metadata.update(dict(f['resp_metadata'].attrs))
            if 'ekg_metadata' in f:
                metadata.update(dict(f['ekg_metadata'].attrs))
            if 'metadata' in f:
                metadata.update(dict(f['metadata'].attrs))

            # Estimate sampling frequency
            if 'sampling_frequency' in metadata:
                fs = metadata['sampling_frequency']
            else:
                duration_sec = metadata.get('duration_sec', None)
                fs = len(resp) / duration_sec if duration_sec else 20000.0

            duration_sec = metadata.get('duration_sec', len(resp) / fs)

    except Exception as e:
        print(f"Error loading {h5_file}: {e}")
        return None, None, None, None

    # Pre-filter before downsampling
    nyquist = fs / 2
    norm_cutoff = (target_rate / 2) / nyquist
    b, a = butter(N=4, Wn=norm_cutoff, btype='low')
    filtered_resp = filtfilt(b, a, resp)

    # Downsample
    downsample_factor = int(fs // target_rate)
    downsampled = resample_poly(filtered_resp, up=1, down=downsample_factor)

    # Bandpass filter with neurokit
    rsp_cleaned = nk.signal_filter(
        downsampled,
        lowcut=0.1,
        highcut=20,
        method="butterworth",
        sampling_rate=target_rate,
        order=2
    )

    # Generate matching time vector
    time_vector = np.arange(len(rsp_cleaned)) / target_rate

    return rsp_cleaned, time_vector, target_rate, metadata


In [10]:
def get_sniff_respiratory_rate(signal, time, sniff_start, sniff_end, sampling_rate=100):
    sniff_mask = (time >= sniff_start) & (time < sniff_end)
    signal_sniff = signal[sniff_mask]

    peaks, _ = find_peaks(signal_sniff, distance=sampling_rate * 0.0833)  # 0.0833 s = 12 Hz
    duration = sniff_end - sniff_start
    rate = len(peaks) / duration if duration > 0 else np.nan

    return rate


In [58]:
import numpy as np
import pandas as pd

def process_all_trials(resp_paths, boris_paths, rank_map, duration_threshold=0.3):
    """
    Loops through each trial in your respiration & BORIS dictionaries.
    Computes mean respiration rate per behavior window and merges metadata.
    """
    all_trials = []

    for trial, h5_path in resp_paths.items():
        if trial not in boris_paths:
            print(f"⚠️ Skipping {trial} — no BORIS file found.")
            continue

        print(f"Processing {trial}...")

        # --- Load respiration ---
        signal, time, fs, meta = load_clean_resp_signal(h5_path)
        if signal is None:
            print(f"❌ Resp load failed for {trial}")
            continue

        # --- Compute session-wide rate ---
        session_rate = get_sniff_respiratory_rate(
            signal=signal,
            time=time,
            sniff_start=time[0],
            sniff_end=time[-1],
            sampling_rate=fs
        )
        print(f"Session-wide respiratory rate for {trial}: {session_rate:.3f} Hz")

        # --- Load BORIS ---
        boris_df = load_clean_boris(boris_paths[trial])
        if boris_df.empty:
            print(f"⚠️ BORIS empty for {trial}")
            continue

        # --- Compute rate per behavior window ---
        boris_df["MeanRate"] = boris_df.apply(
            lambda row: get_sniff_respiratory_rate(
                signal, time, row["Start"], row["Stop"], sampling_rate=fs
            ),
            axis=1
        )
        
        # --- Filter short bouts ---
        pre_len = len(boris_df)
        boris_df = boris_df[boris_df["Duration"] >= duration_threshold].copy()
        post_len = len(boris_df)
        if pre_len != post_len:
            print(f"   → Filtered {pre_len - post_len} short bouts (<{duration_threshold}s)")

        # --- Add metadata ---
        subj = "_".join(trial.split("_")[1:3])   # e.g. RI1_3_6 → 3_6
        condition = trial.split("_")[0]          # e.g. RI1, RI2, CM
        rank = rank_map.get(subj, np.nan)

        boris_df["Trial"] = trial
        boris_df["Subject"] = subj
        boris_df["Condition"] = condition
        boris_df["Rank"] = rank
        boris_df["SessionRate"] = session_rate  # 👈 ADDED HERE

        # --- Append ---
        all_trials.append(boris_df.copy())  # 👈 make sure it keeps new col

        # --- Append ---
        all_trials.append(boris_df)

    # Combine everything into a master dataframe
    if not all_trials:
        print("⚠️ No valid trials processed.")
        return pd.DataFrame()

    master_df = pd.concat(all_trials, ignore_index=True)
    print(f"✅ Combined {len(master_df)} behavior windows across {len(all_trials)} trials.")
    return master_df


In [59]:
# ======================================================
# 🧠 Define rank map (Dominant vs Subordinate)
# ======================================================
rank_map = {
    "1_1": "Subordinate", "1_2": "Dominant",
    "2_3": "Subordinate", "2_4": "Dominant",
    "3_5": "Subordinate", "3_6": "Dominant",
    "4_7": "Subordinate", "4_8": "Dominant"
}

# ======================================================
# 🚀 Process all trials and combine into one DataFrame
# ======================================================
master_df = process_all_trials(resp_paths, boris_paths, rank_map)
print(master_df.head())


print("\n✅ Master DataFrame created!")
print(master_df.head())


# Optionally save
master_df.to_csv(r"E:\Aim1\AIM1\processed_behavior_resp_rates.csv", index=False)
print("💾 Saved to processed_behavior_resp_rates.csv")


Processing RI1_3_6...
Session-wide respiratory rate for RI1_3_6: 7.270 Hz
   → Filtered 1 short bouts (<0.3s)
Processing RI2_3_6...
Session-wide respiratory rate for RI2_3_6: 5.933 Hz
   → Filtered 10 short bouts (<0.3s)
Processing RI1_4_7...
Session-wide respiratory rate for RI1_4_7: 7.564 Hz
   → Filtered 2 short bouts (<0.3s)
Processing RI2_4_7...
Session-wide respiratory rate for RI2_4_7: 6.551 Hz
Processing RI1_2_3...
Session-wide respiratory rate for RI1_2_3: 7.286 Hz
Processing RI2_2_3...
Session-wide respiratory rate for RI2_2_3: 6.674 Hz
Processing RI1_4_8...
Session-wide respiratory rate for RI1_4_8: 7.410 Hz
   → Filtered 5 short bouts (<0.3s)
Processing RI2_4_8...
Session-wide respiratory rate for RI2_4_8: 5.776 Hz
Processing RI1_1_1...
Session-wide respiratory rate for RI1_1_1: 7.433 Hz
   → Filtered 1 short bouts (<0.3s)
Processing RI2_1_1...
Session-wide respiratory rate for RI2_1_1: 6.009 Hz
Processing RI1_1_2...
Session-wide respiratory rate for RI1_1_2: 7.555 Hz
   → 

In [60]:
master_df


,Behavior,Subject,Start,Stop,Duration,MeanRate,Trial,Condition,Rank,SessionRate
0,facial sniffing,3_6,11.966,13.000,1.034,6.769826,RI1_3_6,RI1,Dominant,7.270376
1,anogenital sniffing,3_6,14.483,16.690,2.207,8.608971,RI1_3_6,RI1,Dominant,7.270376
2,anogenital sniffing,3_6,19.069,19.483,0.414,4.830918,RI1_3_6,RI1,Dominant,7.270376
3,anogenital sniffing,3_6,21.379,21.724,0.345,11.594203,RI1_3_6,RI1,Dominant,7.270376
4,anogenital sniffing,3_6,25.034,27.552,2.518,8.339952,RI1_3_6,RI1,Dominant,7.270376
...,...,...,...,...,...,...,...,...,...,...
665,anogenital sniffing,2_4,30.133,30.533,0.400,7.500000,RI2_2_4,RI2,Dominant,6.259910
666,anogenital sniffing,2_4,47.400,47.733,0.333,9.009009,RI2_2_4,RI2,Dominant,6.259910
667,facial sniffing,2_4,77.400,79.667,2.267,9.263344,RI2_2_4,RI2,Dominant,6.259910
668,facial sniffing,3_5,4.067,4.600,0.533,9.380863,RI2_3_5,RI2,Subordinate,5.878217


In [61]:
valence_map = {
    "RI1": "Positive",
    "RI2": "Negative",
    "CM": "Cagemate"
}


In [62]:
# Combine your different condition dictionaries
all_resp_paths = {}
all_resp_paths.update(resp_paths)      # RI1, RI2
all_resp_paths.update(resp_paths_cm)   # CM

all_boris_paths = {}
all_boris_paths.update(boris_paths)
all_boris_paths.update(boris_paths_cm)


In [63]:
# Combine your different condition dictionaries
all_resp_paths = {}
all_resp_paths.update(resp_paths)      # RI1, RI2
all_resp_paths.update(resp_paths_cm)   # CM

all_boris_paths = {}
all_boris_paths.update(boris_paths)
all_boris_paths.update(boris_paths_cm)


In [64]:
master_df_all = process_all_trials(all_resp_paths, all_boris_paths, rank_map)


Processing RI1_3_6...
Session-wide respiratory rate for RI1_3_6: 7.270 Hz
   → Filtered 1 short bouts (<0.3s)
Processing RI2_3_6...
Session-wide respiratory rate for RI2_3_6: 5.933 Hz
   → Filtered 10 short bouts (<0.3s)
Processing RI1_4_7...
Session-wide respiratory rate for RI1_4_7: 7.564 Hz
   → Filtered 2 short bouts (<0.3s)
Processing RI2_4_7...
Session-wide respiratory rate for RI2_4_7: 6.551 Hz
Processing RI1_2_3...
Session-wide respiratory rate for RI1_2_3: 7.286 Hz
Processing RI2_2_3...
Session-wide respiratory rate for RI2_2_3: 6.674 Hz
Processing RI1_4_8...
Session-wide respiratory rate for RI1_4_8: 7.410 Hz
   → Filtered 5 short bouts (<0.3s)
Processing RI2_4_8...
Session-wide respiratory rate for RI2_4_8: 5.776 Hz
Processing RI1_1_1...
Session-wide respiratory rate for RI1_1_1: 7.433 Hz
   → Filtered 1 short bouts (<0.3s)
Processing RI2_1_1...
Session-wide respiratory rate for RI2_1_1: 6.009 Hz
Processing RI1_1_2...
Session-wide respiratory rate for RI1_1_2: 7.555 Hz
   → 

In [65]:
master_df

,Behavior,Subject,Start,Stop,Duration,MeanRate,Trial,Condition,Rank,SessionRate
0,facial sniffing,3_6,11.966,13.000,1.034,6.769826,RI1_3_6,RI1,Dominant,7.270376
1,anogenital sniffing,3_6,14.483,16.690,2.207,8.608971,RI1_3_6,RI1,Dominant,7.270376
2,anogenital sniffing,3_6,19.069,19.483,0.414,4.830918,RI1_3_6,RI1,Dominant,7.270376
3,anogenital sniffing,3_6,21.379,21.724,0.345,11.594203,RI1_3_6,RI1,Dominant,7.270376
4,anogenital sniffing,3_6,25.034,27.552,2.518,8.339952,RI1_3_6,RI1,Dominant,7.270376
...,...,...,...,...,...,...,...,...,...,...
665,anogenital sniffing,2_4,30.133,30.533,0.400,7.500000,RI2_2_4,RI2,Dominant,6.259910
666,anogenital sniffing,2_4,47.400,47.733,0.333,9.009009,RI2_2_4,RI2,Dominant,6.259910
667,facial sniffing,2_4,77.400,79.667,2.267,9.263344,RI2_2_4,RI2,Dominant,6.259910
668,facial sniffing,3_5,4.067,4.600,0.533,9.380863,RI2_3_5,RI2,Subordinate,5.878217


In [66]:
def add_metadata_columns(df):
    df = df.copy()
    # Parse base condition (RI1, RI2, CM)
    df["BaseCond"] = df["Condition"].apply(lambda c: c.split("_")[0] if "_" in c else c)
    df["Valence"] = df["BaseCond"].map(valence_map)
    return df

master_df_all = add_metadata_columns(master_df_all)


In [67]:
master_df_all

,Behavior,Subject,Start,Stop,Duration,MeanRate,Trial,Condition,Rank,SessionRate,BaseCond,Valence
0,facial sniffing,3_6,11.966,13.000,1.034,6.769826,RI1_3_6,RI1,Dominant,7.270376,RI1,Positive
1,anogenital sniffing,3_6,14.483,16.690,2.207,8.608971,RI1_3_6,RI1,Dominant,7.270376,RI1,Positive
2,anogenital sniffing,3_6,19.069,19.483,0.414,4.830918,RI1_3_6,RI1,Dominant,7.270376,RI1,Positive
3,anogenital sniffing,3_6,21.379,21.724,0.345,11.594203,RI1_3_6,RI1,Dominant,7.270376,RI1,Positive
4,anogenital sniffing,3_6,25.034,27.552,2.518,8.339952,RI1_3_6,RI1,Dominant,7.270376,RI1,Positive
...,...,...,...,...,...,...,...,...,...,...,...,...
981,facial sniffing,4_8,375.067,377.267,2.200,7.272727,CM_4_8_sub4_7,CM,Dominant,6.970979,CM,Cagemate
982,facial sniffing,4_8,425.533,427.766,2.233,8.060905,CM_4_8_sub4_7,CM,Dominant,6.970979,CM,Cagemate
983,body sniffing,4_8,427.767,429.467,1.700,8.823529,CM_4_8_sub4_7,CM,Dominant,6.970979,CM,Cagemate
984,body sniffing,4_8,433.800,434.867,1.067,8.434864,CM_4_8_sub4_7,CM,Dominant,6.970979,CM,Cagemate
